# Week 25, Day 4: Project: the interface

Companion notebook. The interactive figures, X-ray views, quizzes and auto-marked exercises are on the [lesson page](https://peter-adepoju.github.io/python-atlas/days/w025d4.html). Use this notebook to keep your own copy of the code.

**How to use it:** run each cell with Shift+Enter. Change the code and run it again: that's how you learn.

In [ ]:
# Run this cell once. It lets you check exercise answers inside this notebook.
# Workflow: run your exercise cell, then immediately run the "check" cell below it.
import io, builtins

def check(tests_src):
    """Run an exercise's tests against the cell you ran just before this one."""
    ip = get_ipython()
    code = ip.user_ns["In"][-2]
    def run(feed=(), echo=False):
        out = io.StringIO(); q = [str(x) for x in feed]; real_in, real_print = builtins.input, builtins.print
        def fake_input(p=""):
            if not q:
                raise AssertionError("The program asked for more input than the test gave it.")
            v = q.pop(0); out.write(str(p) + (v + "\n" if echo else "")); return v
        def fake_print(*a, **k):
            k.setdefault("file", out); real_print(*a, **k)
        builtins.input, builtins.print = fake_input, fake_print
        try:
            exec(code, {"__name__": "__main__"})
        finally:
            builtins.input, builtins.print = real_in, real_print
        return out.getvalue()
    norm = lambda t: "\n".join(l.rstrip() for l in str(t).rstrip().splitlines())
    def expect_output(want, inputs=(), echo=False):
        got = run(inputs, echo)
        assert norm(got) == norm(want), f"expected {norm(want)!r} but got {norm(got)!r}"
    import re
    uses = lambda *names: all(re.search(r"\b" + re.escape(n) + r"\b", code) for n in names)
    def run_with(subs, feed=(), echo=False):
        nonlocal code
        saved = code
        for old, new in subs.items():
            code = code.replace(old, new, 1)
        try:
            return run(feed, echo)
        finally:
            code = saved
    env = dict(ip.user_ns, run=run, norm=norm, source=code, expect_output=expect_output, uses=uses, run_with=run_with)
    try:
        env["output"] = run()
    except AssertionError:
        env["output"] = ""
    exec(tests_src, env)
    tests = [(k, v) for k, v in env.items() if k.startswith("test_") and callable(v)]
    ok = 0
    for name, fn in tests:
        label = (fn.__doc__ or name[5:].replace("_", " ")).strip()
        try:
            fn(); ok += 1; print("✓", label)
        except AssertionError as e:
            print("✗", label + ":", e or "not quite yet")
    print(f"{ok} of {len(tests)} checks passed.")


These are the files this page's programs use.

In [ ]:
# Run this cell first: it creates the files used below.
with open('money.py', 'w', encoding='utf-8') as f:
    f.write('"""Reading and showing money and categories."""\nimport re\n\nAMOUNT = re.compile(r"£?\\s*(\\d{1,3}(?:,\\d{3})+|\\d+)(?:\\.(\\d{1,2}))?")\n\n\ndef parse_amount(text):\n    """Text such as \'£1,250.50\' as whole pence. Raises ValueError with a readable message."""\n    text = text.strip()\n    if not text.lstrip("£").strip():\n        raise ValueError("missing amount")\n    m = AMOUNT.fullmatch(text)\n    if m is None:\n        raise ValueError(f"not an amount: {text!r}")\n    total = int(m.group(1).replace(",", "")) * 100 + int((m.group(2) or "").ljust(2, "0"))\n    if total == 0:\n        raise ValueError("amount must be more than £0")\n    return total\n\n\ndef format_pence(pence):\n    """1250 as \'£12.50\', -350 as \'-£3.50\'."""\n    sign = "-" if pence < 0 else ""\n    pounds, rest = divmod(abs(pence), 100)\n    return f"{sign}£{pounds:,}.{rest:02d}"\n\n\ndef clean_category(text):\n    """\' Eating   Out \' as \'eating out\'. Raises ValueError if it isn\'t words of letters."""\n    shown = text.strip()\n    if not shown:\n        raise ValueError("missing category")\n    cleaned = " ".join(shown.lower().split())\n    if len(cleaned) > 20 or not re.fullmatch(r"[a-z]+( [a-z]+)*", cleaned):\n        raise ValueError(f"bad category: {shown!r}")\n    return cleaned\n')
with open('dates.py', 'w', encoding='utf-8') as f:
    f.write('"""Reading dates and months that people type, and stepping through months."""\nimport re\nfrom datetime import datetime, timedelta\n\nFORMATS = ["%Y-%m-%d", "%d/%m/%Y"]\n\n\ndef parse_date(text, today):\n    """\'2026-11-02\', \'02/11/2026\', \'today\', \'yesterday\' or \'\' as ISO text. No future dates."""\n    shown = text.strip()\n    word = shown.lower()\n    if word in ("", "today"):\n        when = today\n    elif word == "yesterday":\n        when = today - timedelta(days=1)\n    else:\n        when = None\n        for fmt in FORMATS:\n            try:\n                when = datetime.strptime(word, fmt).date()\n                break\n            except ValueError:\n                pass\n        if when is None:\n            raise ValueError(f"not a date: {shown!r}")\n    if when > today:\n        raise ValueError(f"date is in the future: {when.isoformat()}")\n    return when.isoformat()\n\n\ndef parse_month(text):\n    """\'2026-11\' or \'11/2026\' as \'2026-11\'. Raises ValueError otherwise."""\n    shown = text.strip()\n    m = re.fullmatch(r"(\\d{4})-(\\d{2})", shown)\n    if m:\n        year, month = int(m.group(1)), int(m.group(2))\n    else:\n        m = re.fullmatch(r"(\\d{1,2})/(\\d{4})", shown)\n        if not m:\n            raise ValueError(f"not a month: {shown!r}")\n        month, year = int(m.group(1)), int(m.group(2))\n    if not 1 <= month <= 12:\n        raise ValueError(f"not a month: {shown!r}")\n    return f"{year}-{month:02d}"\n\n\ndef next_month(month):\n    """\'2026-12\' gives \'2027-01\'."""\n    year, m = int(month[:4]), int(month[5:])\n    if m == 12:\n        return f"{year + 1}-01"\n    return f"{year}-{m + 1:02d}"\n')
with open('core.py', 'w', encoding='utf-8') as f:
    f.write('"""The expense tracker\'s data and the operations on it."""\nfrom money import parse_amount, clean_category\nfrom dates import parse_date, next_month\n\n\ndef new_data():\n    return {"expenses": [], "budgets": {}, "next_id": 1}\n\n\ndef add_expense(data, amount_text, category_text, note, date_text, today):\n    """Check every field, then add the expense, or raise one ValueError listing every problem."""\n    problems = []\n    amount = category = when = None\n    try:\n        amount = parse_amount(amount_text)\n    except ValueError as e:\n        problems.append(str(e))\n    try:\n        category = clean_category(category_text)\n    except ValueError as e:\n        problems.append(str(e))\n    try:\n        when = parse_date(date_text, today)\n    except ValueError as e:\n        problems.append(str(e))\n    note = note.strip()\n    if len(note) > 60:\n        problems.append("note is longer than 60 characters")\n    if problems:\n        raise ValueError("; ".join(problems))\n    expense = {"id": data["next_id"], "date": when, "amount": amount, "category": category, "note": note}\n    data["expenses"].append(expense)\n    data["next_id"] += 1\n    return expense\n\n\ndef remove_expense(data, expense_id):\n    """Remove and return the expense with this id. KeyError if there isn\'t one."""\n    for i, e in enumerate(data["expenses"]):\n        if e["id"] == expense_id:\n            return data["expenses"].pop(i)\n    raise KeyError(f"no expense #{expense_id}")\n\n\ndef filter_expenses(data, month=None, category=None):\n    """A new list of the matching expenses, newest first (by date, then id)."""\n    chosen = [e for e in data["expenses"]\n              if (month is None or e["date"][:7] == month)\n              and (category is None or e["category"] == category)]\n    return sorted(chosen, key=lambda e: (e["date"], e["id"]), reverse=True)\n\n\ndef total(expenses):\n    return sum(e["amount"] for e in expenses)\n\n\ndef by_category(expenses):\n    """(category, pence) pairs, largest first, alphabetical for ties."""\n    totals = {}\n    for e in expenses:\n        totals[e["category"]] = totals.get(e["category"], 0) + e["amount"]\n    return sorted(totals.items(), key=lambda pair: (-pair[1], pair[0]))\n\n\ndef month_summary(data, month):\n    """Count, total, a row per category with spending or a budget, and the categories over budget."""\n    expenses = filter_expenses(data, month=month)\n    spent = dict(by_category(expenses))\n    rows = []\n    for category in set(spent) | set(data["budgets"]):\n        budget = data["budgets"].get(category)\n        used = spent.get(category, 0)\n        rows.append({"category": category, "spent": used, "budget": budget,\n                     "left": None if budget is None else budget - used})\n    rows.sort(key=lambda r: (-r["spent"], r["category"]))\n    return {"month": month, "count": len(expenses), "total": total(expenses), "rows": rows,\n            "over": [r["category"] for r in rows if r["left"] is not None and r["left"] < 0]}\n\n\ndef monthly_totals(data):\n    """(month, pence) for every month from the first with spending to the last, gaps as 0."""\n    totals = {}\n    for e in data["expenses"]:\n        totals[e["date"][:7]] = totals.get(e["date"][:7], 0) + e["amount"]\n    if not totals:\n        return []\n    out = []\n    month, last = min(totals), max(totals)\n    while month <= last:\n        out.append((month, totals.get(month, 0)))\n        month = next_month(month)\n    return out\n\n\ndef set_budget(data, category_text, amount_text):\n    """Check both, then set the category\'s monthly budget. Returns (category, pence)."""\n    problems = []\n    category = amount = None\n    try:\n        category = clean_category(category_text)\n    except ValueError as e:\n        problems.append(str(e))\n    try:\n        amount = parse_amount(amount_text)\n    except ValueError as e:\n        problems.append(str(e))\n    if problems:\n        raise ValueError("; ".join(problems))\n    data["budgets"][category] = amount\n    return category, amount\n\n\ndef remove_budget(data, category_text):\n    """Remove a category\'s budget and return it. KeyError if it has none."""\n    category = clean_category(category_text)\n    if category not in data["budgets"]:\n        raise KeyError(f"no budget for {category!r}")\n    return data["budgets"].pop(category)\n')
with open('storage.py', 'w', encoding='utf-8') as f:
    f.write('"""Saving and loading the data, and importing and exporting CSV."""\nimport csv\nimport io\nimport json\nimport os\nimport re\nfrom collections import Counter\nfrom datetime import datetime\n\nfrom core import new_data\nfrom money import parse_amount\n\nKEYS = ["id", "date", "amount", "category", "note"]\n\n\ndef load_data(path):\n    """The data saved at path, or new_data() if there\'s no file. ValueError if it\'s damaged."""\n    if not os.path.exists(path):\n        return new_data()\n    with open(path, encoding="utf-8") as f:\n        try:\n            data = json.load(f)\n        except json.JSONDecodeError as e:\n            raise ValueError(f"{path} is damaged: {e.msg} at line {e.lineno}") from None\n    if not (isinstance(data, dict) and isinstance(data.get("expenses"), list)\n            and isinstance(data.get("budgets"), dict) and isinstance(data.get("next_id"), int)):\n        raise ValueError(f"{path} is not a spend data file")\n    for e in data["expenses"]:\n        if not (isinstance(e, dict) and list(e) == KEYS and isinstance(e["amount"], int)):\n            raise ValueError(f"{path} has a damaged expense: {e!r}")\n    return data\n\n\ndef save_data(path, data):\n    """Save safely: write a temporary file, then replace the old file with it in one step."""\n    tmp = path + ".tmp"\n    try:\n        with open(tmp, "w", encoding="utf-8") as f:\n            json.dump(data, f, indent=2, ensure_ascii=False)\n        os.replace(tmp, path)\n    finally:\n        if os.path.exists(tmp):\n            os.remove(tmp)\n\n\nSTATEMENT_AMOUNT = re.compile(r"(-?)(£?[\\d,]+(?:\\.\\d{1,2})?)")\n\n\ndef read_statement(text):\n    """A bank CSV (Date, Description, Amount) as (rows, problems). Only money out is kept."""\n    rows, problems = [], []\n    reader = csv.DictReader(io.StringIO(text))\n    for row in reader:\n        line = reader.line_num\n        try:\n            when = datetime.strptime(row["Date"].strip(), "%d/%m/%Y").date().isoformat()\n        except ValueError:\n            problems.append(f"line {line}: bad date {row[\'Date\'].strip()!r}")\n            continue\n        m = STATEMENT_AMOUNT.fullmatch(row["Amount"].strip())\n        try:\n            if m is None:\n                raise ValueError\n            pence = parse_amount(m.group(2))\n        except ValueError:\n            problems.append(f"line {line}: bad amount {row[\'Amount\'].strip()!r}")\n            continue\n        if m.group(1) != "-":\n            continue\n        rows.append({"date": when, "amount": pence, "description": " ".join(row["Description"].split())})\n    return rows, problems\n\n\nRULES = [\n    (r"tesco|sainsbury|aldi|lidl|co-?op", "food"),\n    (r"\\btfl\\b|trainline|uber|bus", "travel"),\n    (r"netflix|spotify|cinema", "fun"),\n    (r"rent|lettings", "rent"),\n]\n\n\ndef guess_category(description, rules=RULES):\n    for pattern, category in rules:\n        if re.search(pattern, description, re.IGNORECASE):\n            return category\n    return "other"\n\n\ndef import_statement(data, text, rules=RULES):\n    rows, problems = read_statement(text)\n    existing = Counter((e["date"], e["amount"], e["note"]) for e in data["expenses"])\n    added = skipped = 0\n    for r in rows:\n        note = r["description"][:60]\n        key = (r["date"], r["amount"], note)\n        if existing[key] > 0:\n            existing[key] -= 1\n            skipped += 1\n            continue\n        data["expenses"].append({"id": data["next_id"], "date": r["date"], "amount": r["amount"],\n                                 "category": guess_category(r["description"], rules), "note": note})\n        data["next_id"] += 1\n        added += 1\n    return {"added": added, "skipped": skipped, "problems": problems}\n\n\ndef export_csv(data, path):\n    with open(path, "w", newline="", encoding="utf-8") as f:\n        w = csv.writer(f)\n        w.writerow(["id", "date", "amount", "category", "note"])\n        for e in sorted(data["expenses"], key=lambda e: (e["date"], e["id"])):\n            w.writerow([e["id"], e["date"], f"{e[\'amount\'] // 100}.{e[\'amount\'] % 100:02d}", e["category"], e["note"]])\n')
with open('statement.csv', 'w', encoding='utf-8') as f:
    f.write('Date,Description,Amount,Balance\n02/11/2026,TESCO STORES 2041,-62.10,1702.15\n02/11/2026,TFL TRAVEL CH,-4.20,1764.25\n02/11/2026,TFL TRAVEL CH,-4.20,1768.45\n01/11/2026,SALARY,"2,500.00",1772.65\n01/11/2026,CITY LETTINGS RENT,"-3,200.00",-727.35\n31/10/2026,NETFLIX.COM,-10.99,2472.65\n31/10/2026,CORNER CAFE,-3.80,2483.64\n30/10/2026,  PRET   A MANGER ,-6.45,2487.44\n3O/10/2026,UBER *TRIP,-14.00,2493.89\n29/10/2026,AMAZON MKTPLACE,-1O.00,2507.89\n')

### Exercise: Step 21: describe and tabulate

Write `describe(e)`, returning `#14: £12.50 food (lunch with Tola) on 2026-11-02` (no brackets when the note is empty: `#1: £3,200.00 rent on 2026-11-01`), and `expense_table(expenses)`, returning a list of lines:

- the header `f"{'#':>4}  {'Date':<10}  {'Amount':>11}  {'Category':<12}  Note"`;
- one line per expense in the same columns, with the amount formatted by `format_pence`, and trailing spaces removed (for an empty note);
- a total line: 4 spaces, two spaces, `Total` in the date column, the total in the amount column, two spaces, then `1 expense` or `N expenses`;
- or, for no expenses at all, just `["No expenses."]`.

In [ ]:
from money import format_pence
from core import total

def describe(e):
    ___

def expense_table(expenses):
    ___

xs = [{"id": 7, "date": "2026-11-02", "amount": 6210, "category": "food", "note": "TESCO STORES 2041"},
      {"id": 1, "date": "2026-11-01", "amount": 320000, "category": "rent", "note": ""}]
print(describe(xs[0]))
print("\n".join(expense_table(xs)))

In [ ]:
# Check: run the exercise cell above first, then this one.
check("XS = [{\"id\": 7, \"date\": \"2026-11-02\", \"amount\": 6210, \"category\": \"food\", \"note\": \"TESCO STORES 2041\"},\n      {\"id\": 1, \"date\": \"2026-11-01\", \"amount\": 320000, \"category\": \"rent\", \"note\": \"\"}]\ndef test_describe():\n    \"describe\"\n    assert describe(XS[0]) == \"#7: \u00a362.10 food (TESCO STORES 2041) on 2026-11-02\"\n    assert describe(XS[1]) == \"#1: \u00a33,200.00 rent on 2026-11-01\"\ndef test_table():\n    \"the table\"\n    assert expense_table(XS) == [\n        \"   #  Date             Amount  Category      Note\",\n        \"   7  2026-11-02       \u00a362.10  food          TESCO STORES 2041\",\n        \"   1  2026-11-01    \u00a33,200.00  rent\",\n        \"      Total         \u00a33,262.10  2 expenses\",\n    ], expense_table(XS)\ndef test_one_and_none():\n    \"one expense, and none\"\n    assert expense_table(XS[:1])[-1] == \"      Total            \u00a362.10  1 expense\"\n    assert expense_table([]) == [\"No expenses.\"]")

### Exercise: Step 22: the summary

Write `month_name(month)` (`"2026-11"` as `"November 2026"`) and `summary_lines(s)`, taking `month_summary`'s result and returning:

- the title: `November 2026: 6 expenses, £3,329.40` (`1 expense` for one);
- for each row: two spaces, the category in 12, two spaces, the spent amount in 11 (right-aligned); then, only if there's a budget, `f"  of {budget:>9}  "` followed by `OVER by £5.90` (when `left` is below 0) or `£41.60 left`.

In [ ]:
from datetime import datetime
from money import format_pence

def month_name(month):
    ___

def summary_lines(s):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("S = {\"month\": \"2026-11\", \"count\": 6, \"total\": 332940, \"rows\": [\n    {\"category\": \"rent\", \"spent\": 320000, \"budget\": None, \"left\": None},\n    {\"category\": \"food\", \"spent\": 6590, \"budget\": 6000, \"left\": -590},\n    {\"category\": \"travel\", \"spent\": 840, \"budget\": 5000, \"left\": 4160},\n    {\"category\": \"fun\", \"spent\": 0, \"budget\": 3000, \"left\": 3000}], \"over\": [\"food\"]}\ndef test_month_name():\n    \"month_name\"\n    assert month_name(\"2026-11\") == \"November 2026\" and month_name(\"2027-01\") == \"January 2027\"\ndef test_lines():\n    \"the summary\"\n    assert summary_lines(S) == [\n        \"November 2026: 6 expenses, \u00a33,329.40\",\n        \"  rent            \u00a33,200.00\",\n        \"  food               \u00a365.90  of    \u00a360.00  OVER by \u00a35.90\",\n        \"  travel              \u00a38.40  of    \u00a350.00  \u00a341.60 left\",\n        \"  fun                 \u00a30.00  of    \u00a330.00  \u00a330.00 left\",\n    ], summary_lines(S)\ndef test_one_and_empty():\n    \"one expense; an empty month; exactly on budget\"\n    one = {\"month\": \"2026-12\", \"count\": 1, \"total\": 500, \"over\": [],\n           \"rows\": [{\"category\": \"fun\", \"spent\": 500, \"budget\": 500, \"left\": 0}]}\n    assert summary_lines(one) == [\"December 2026: 1 expense, \u00a35.00\", \"  fun                 \u00a35.00  of     \u00a35.00  \u00a30.00 left\"]\n    assert summary_lines({\"month\": \"2027-02\", \"count\": 0, \"total\": 0, \"rows\": [], \"over\": []}) == [\"February 2027: 0 expenses, \u00a30.00\"]")

### Exercise: Step 23: build the parser

Write `build_parser()` returning an `ArgumentParser` with `prog="spend"`, the `--file` option, and the seven subcommands in the table, using `dest="command"` and `required=True` for the subparsers. Give every subcommand a `help` string. An optional positional argument uses `nargs="?"`; give the `add` command's `note` a default of `""`.

In [ ]:
import argparse

def build_parser():
    ___

print(build_parser().parse_args(["add", "12.50", "food", "lunch", "--date", "yesterday"]))
print(build_parser().parse_args(["--file", "test.json", "budget"]))

In [ ]:
# Check: run the exercise cell above first, then this one.
check("import contextlib as _cl, io as _io\ndef parse(argv):\n    return vars(build_parser().parse_args(argv))\ndef test_add():\n    \"add\"\n    assert parse([\"add\", \"12.50\", \"food\", \"lunch\", \"--date\", \"yesterday\"]) == \\\n        {\"file\": \"spend.json\", \"command\": \"add\", \"amount\": \"12.50\", \"category\": \"food\", \"note\": \"lunch\", \"date\": \"yesterday\"}\n    assert parse([\"add\", \"3\", \"fun\"])[\"note\"] == \"\" and parse([\"add\", \"3\", \"fun\"])[\"date\"] == \"\"\ndef test_others():\n    \"the other commands\"\n    assert parse([\"remove\", \"14\"]) == {\"file\": \"spend.json\", \"command\": \"remove\", \"id\": 14}\n    assert parse([\"list\"]) == {\"file\": \"spend.json\", \"command\": \"list\", \"month\": None, \"category\": None}\n    assert parse([\"list\", \"--month\", \"2026-11\", \"--category\", \"food\"])[\"category\"] == \"food\"\n    assert parse([\"summary\"])[\"month\"] is None\n    assert parse([\"--file\", \"t.json\", \"budget\"]) == {\"file\": \"t.json\", \"command\": \"budget\", \"category\": None, \"amount\": None}\n    assert parse([\"budget\", \"food\", \"none\"])[\"amount\"] == \"none\"\n    assert parse([\"import\", \"s.csv\"])[\"statement\"] == \"s.csv\" and parse([\"export\", \"o.csv\"])[\"out\"] == \"o.csv\"\ndef test_rejects():\n    \"nonsense exits with status 2\"\n    for argv in ([], [\"fly\"], [\"remove\", \"two\"], [\"add\", \"5\"], [\"list\", \"extra\"]):\n        err = _io.StringIO()\n        try:\n            with _cl.redirect_stderr(err):\n                build_parser().parse_args(argv)\n            assert False, f\"{argv} should be rejected\"\n        except SystemExit as e:\n            assert e.code == 2, argv\ndef test_help():\n    \"every subcommand has help\"\n    out = _io.StringIO()\n    try:\n        with _cl.redirect_stdout(out):\n            build_parser().parse_args([\"--help\"])\n    except SystemExit:\n        pass\n    text = out.getvalue()\n    assert text.startswith(\"usage: spend\") and all(c in text for c in (\"add\", \"remove\", \"list\", \"summary\", \"budget\", \"import\", \"export\"))\n    assert text.count(\"{add,remove,list,summary,budget,import,export}\") >= 1, \"add the subcommands in the table's order\"")

### Exercise: Step 24: the commands

Write the command functions and `main(argv=None, today=None)`. The starter has everything from Steps 21 to 23. The outputs:

- `add`: `Added #14: ...` (using `describe`); `remove`: `Removed #14: ...`;
- `list`: the `expense_table` of the filtered expenses (month through `parse_month`, category through `clean_category`, when given);
- `summary`: the `summary_lines` for the month given, or today's month;
- `budget` with no category: each budget as `f"  {category:<12}  {format_pence(pence):>11} a month"`, alphabetically, or `No budgets.`; with a category and an amount: `Budget for food: £60.00 a month`; with `none`: `Removed the budget for food`; with a category but no amount: a `ValueError("give an amount, or 'none' to remove the budget")`;
- `import`: `Imported 7, skipped 0 already there`, then `  could not read line 10: ...` for each problem; it changed the data only if something was added;
- `export`: `Exported 10 expenses to out.csv`.

`main` parses the arguments, loads the data, runs the command, saves if it returns `True`, and returns 0. A `ValueError`, `KeyError` or `OSError` (a missing statement file, say) prints `spend: ` and the message to standard error (for a `KeyError`, its `args[0]`, which has no quotes round it) and returns 1, saving nothing.

In [ ]:
"""spend: a command-line expense tracker."""
import argparse
import sys
from datetime import date, datetime

from core import (add_expense, remove_expense, filter_expenses, total, month_summary,
                  set_budget, remove_budget)
from dates import parse_month
from money import format_pence, clean_category
from storage import load_data, save_data, import_statement, export_csv


def describe(e):
    """'#14: £12.50 food (lunch with Tola) on 2026-11-02'"""
    note = f" ({e['note']})" if e["note"] else ""
    return f"#{e['id']}: {format_pence(e['amount'])} {e['category']}{note} on {e['date']}"


def expense_table(expenses):
    """The lines of a table of expenses, with a total line. No expenses gives one line saying so."""
    if not expenses:
        return ["No expenses."]
    lines = [f"{'#':>4}  {'Date':<10}  {'Amount':>11}  {'Category':<12}  Note"]
    for e in expenses:
        lines.append(f"{e['id']:>4}  {e['date']:<10}  {format_pence(e['amount']):>11}  {e['category']:<12}  {e['note']}".rstrip())
    count = "1 expense" if len(expenses) == 1 else f"{len(expenses)} expenses"
    lines.append(f"{'':>4}  {'Total':<10}  {format_pence(total(expenses)):>11}  {count}")
    return lines


def month_name(month):
    """'2026-11' as 'November 2026'."""
    return datetime.strptime(month, "%Y-%m").strftime("%B %Y")


def summary_lines(s):
    """The lines of a month's summary, from month_summary's result."""
    count = "1 expense" if s["count"] == 1 else f"{s['count']} expenses"
    lines = [f"{month_name(s['month'])}: {count}, {format_pence(s['total'])}"]
    for r in s["rows"]:
        line = f"  {r['category']:<12}  {format_pence(r['spent']):>11}"
        if r["budget"] is not None:
            line += f"  of {format_pence(r['budget']):>9}  "
            if r["left"] < 0:
                line += f"OVER by {format_pence(-r['left'])}"
            else:
                line += f"{format_pence(r['left'])} left"
        lines.append(line)
    return lines


def build_parser():
    parser = argparse.ArgumentParser(prog="spend", description="Track what you spend.")
    parser.add_argument("--file", default="spend.json", help="the data file (default: spend.json)")
    sub = parser.add_subparsers(dest="command", required=True)

    p = sub.add_parser("add", help="record an expense")
    p.add_argument("amount", help="for example 12.50 or £1,250")
    p.add_argument("category", help="for example food")
    p.add_argument("note", nargs="?", default="", help="what it was for")
    p.add_argument("--date", default="", help="YYYY-MM-DD, DD/MM/YYYY, today or yesterday")

    p = sub.add_parser("remove", help="remove an expense by its number")
    p.add_argument("id", type=int)

    p = sub.add_parser("list", help="list expenses, newest first")
    p.add_argument("--month", help="YYYY-MM or MM/YYYY")
    p.add_argument("--category")

    p = sub.add_parser("summary", help="a month's spending by category")
    p.add_argument("--month", help="YYYY-MM or MM/YYYY (default: this month)")

    p = sub.add_parser("budget", help="show, set or remove monthly budgets")
    p.add_argument("category", nargs="?")
    p.add_argument("amount", nargs="?", help="an amount, or 'none' to remove the budget")

    p = sub.add_parser("import", help="import a bank statement CSV")
    p.add_argument("statement")

    p = sub.add_parser("export", help="export everything as CSV")
    p.add_argument("out")
    return parser


def cmd_add(data, args, today):
    e = add_expense(data, args.amount, args.category, args.note, args.date, today)
    print("Added", describe(e))
    return True

# Write cmd_remove, cmd_list, cmd_summary, cmd_budget, cmd_import and cmd_export here.

COMMANDS = {"add": cmd_add}   # add the others

def main(argv=None, today=None):
    ___

if __name__ == "__main__":
    sys.exit(main())

In [ ]:
# Check: run the exercise cell above first, then this one.
check("import contextlib as _cl, io as _io, json as _json, os as _os\nfrom datetime import date as _date\nT = _date(2026, 11, 2)\ndef go(*argv):\n    out, err = _io.StringIO(), _io.StringIO()\n    with _cl.redirect_stdout(out), _cl.redirect_stderr(err):\n        code = main(list(argv), today=T)\n    return code, out.getvalue(), err.getvalue()\ndef fresh():\n    reset_files()\ndef test_add_remove():\n    \"add and remove, saved to the file\"\n    fresh()\n    assert go(\"add\", \"3,200\", \"Rent\", \"--date\", \"01/11/2026\") == (0, \"Added #1: \u00a33,200.00 rent on 2026-11-01\\n\", \"\")\n    assert go(\"add\", \"12.50\", \"food\", \"lunch\") == (0, \"Added #2: \u00a312.50 food (lunch) on 2026-11-02\\n\", \"\")\n    saved = _json.loads(read_file(\"spend.json\"))\n    assert [e[\"id\"] for e in saved[\"expenses\"]] == [1, 2] and saved[\"next_id\"] == 3\n    assert go(\"remove\", \"1\") == (0, \"Removed #1: \u00a33,200.00 rent on 2026-11-01\\n\", \"\")\n    assert _json.loads(read_file(\"spend.json\"))[\"next_id\"] == 3\ndef test_errors():\n    \"errors go to standard error with status 1, and nothing is saved\"\n    fresh()\n    go(\"add\", \"5\", \"food\")\n    before = read_file(\"spend.json\")\n    assert go(\"add\", \"12.345\", \"Food & Drink\", \"--date\", \"tomorrow\") == \\\n        (1, \"\", \"spend: not an amount: '12.345'; bad category: 'Food & Drink'; not a date: 'tomorrow'\\n\")\n    assert go(\"remove\", \"9\") == (1, \"\", \"spend: no expense #9\\n\")\n    assert go(\"budget\", \"fun\") == (1, \"\", \"spend: give an amount, or 'none' to remove the budget\\n\")\n    assert go(\"import\", \"missing.csv\")[0] == 1\n    assert read_file(\"spend.json\") == before\n    write_file(\"spend.json\", \"{oops\")\n    code, out, err = go(\"list\")\n    assert code == 1 and err.startswith(\"spend: spend.json is damaged\")\ndef test_list_and_summary():\n    \"list and summary\"\n    fresh()\n    go(\"add\", \"3,200\", \"rent\", \"--date\", \"2026-11-01\")\n    go(\"add\", \"62.10\", \"food\", \"shop\")\n    go(\"add\", \"4.20\", \"travel\", \"bus\", \"--date\", \"2026-10-31\")\n    go(\"budget\", \"food\", \"60\")\n    code, out, err = go(\"list\", \"--month\", \"11/2026\")\n    assert code == 0 and out.splitlines() == [\n        \"   #  Date             Amount  Category      Note\",\n        \"   2  2026-11-02       \u00a362.10  food          shop\",\n        \"   1  2026-11-01    \u00a33,200.00  rent\",\n        \"      Total         \u00a33,262.10  2 expenses\"], out\n    assert go(\"list\", \"--category\", \" Travel \")[1].splitlines()[1].startswith(\"   3  2026-10-31\")\n    assert go(\"list\", \"--month\", \"2025-01\") == (0, \"No expenses.\\n\", \"\")\n    code, out, err = go(\"summary\")\n    assert out.splitlines() == [\"November 2026: 2 expenses, \u00a33,262.10\", \"  rent            \u00a33,200.00\",\n                                \"  food               \u00a362.10  of    \u00a360.00  OVER by \u00a32.10\"], out\n    assert go(\"summary\", \"--month\", \"13/2026\") == (1, \"\", \"spend: not a month: '13/2026'\\n\")\ndef test_budgets():\n    \"budget\"\n    fresh()\n    assert go(\"budget\") == (0, \"No budgets.\\n\", \"\")\n    assert go(\"budget\", \"Fun\", \"\u00a330\") == (0, \"Budget for fun: \u00a330.00 a month\\n\", \"\")\n    go(\"budget\", \"food\", \"60\")\n    assert go(\"budget\") == (0, \"  food               \u00a360.00 a month\\n  fun                \u00a330.00 a month\\n\", \"\")\n    assert go(\"budget\", \"fun\", \"None\") == (0, \"Removed the budget for fun\\n\", \"\")\n    assert go(\"budget\", \"fun\", \"none\") == (1, \"\", \"spend: no budget for 'fun'\\n\")\ndef test_import_export():\n    \"import and export\"\n    fresh()\n    code, out, err = go(\"import\", \"statement.csv\")\n    assert code == 0 and out.splitlines() == [\"Imported 7, skipped 0 already there\",\n        \"  could not read line 10: bad date '3O/10/2026'\", \"  could not read line 11: bad amount '-1O.00'\"], out\n    assert go(\"import\", \"statement.csv\")[1].startswith(\"Imported 0, skipped 7\")\n    assert go(\"export\", \"out.csv\") == (0, \"Exported 7 expenses to out.csv\\n\", \"\")\n    assert read_file(\"out.csv\").startswith(\"id,date,amount,category,note\\r\\n7,2026-10-30,6.45,other,PRET A MANGER\")")

> **Why main takes argv and today.** Because `main` takes its inputs as parameters, a test can run the whole program, every layer from parsing to saving, without a terminal and on any date it likes. The real program passes nothing, so it gets the real command line and the real date. It's the same idea as `parse_date(text, today)` on Day 1, applied to the whole program.

### Exercise: Step 25: end-to-end tests

Write tests for `spend.py` that call `main([...], today=...)` with `--file` pointing into `tmp_path`, and check the output with `capsys`. They must pass on the real program and catch all **six** broken versions.

In [ ]:
# Run this cell first: it creates the files used below.
with open('spend.py', 'w', encoding='utf-8') as f:
    f.write('"""spend: a command-line expense tracker."""\nimport argparse\nimport sys\nfrom datetime import date, datetime\n\nfrom core import (add_expense, remove_expense, filter_expenses, total, month_summary,\n                  set_budget, remove_budget)\nfrom dates import parse_month\nfrom money import format_pence, clean_category\nfrom storage import load_data, save_data, import_statement, export_csv\n\n\ndef describe(e):\n    """\'#14: £12.50 food (lunch with Tola) on 2026-11-02\'"""\n    note = f" ({e[\'note\']})" if e["note"] else ""\n    return f"#{e[\'id\']}: {format_pence(e[\'amount\'])} {e[\'category\']}{note} on {e[\'date\']}"\n\n\ndef expense_table(expenses):\n    """The lines of a table of expenses, with a total line. No expenses gives one line saying so."""\n    if not expenses:\n        return ["No expenses."]\n    lines = [f"{\'#\':>4}  {\'Date\':<10}  {\'Amount\':>11}  {\'Category\':<12}  Note"]\n    for e in expenses:\n        lines.append(f"{e[\'id\']:>4}  {e[\'date\']:<10}  {format_pence(e[\'amount\']):>11}  {e[\'category\']:<12}  {e[\'note\']}".rstrip())\n    count = "1 expense" if len(expenses) == 1 else f"{len(expenses)} expenses"\n    lines.append(f"{\'\':>4}  {\'Total\':<10}  {format_pence(total(expenses)):>11}  {count}")\n    return lines\n\n\ndef month_name(month):\n    """\'2026-11\' as \'November 2026\'."""\n    return datetime.strptime(month, "%Y-%m").strftime("%B %Y")\n\n\ndef summary_lines(s):\n    """The lines of a month\'s summary, from month_summary\'s result."""\n    count = "1 expense" if s["count"] == 1 else f"{s[\'count\']} expenses"\n    lines = [f"{month_name(s[\'month\'])}: {count}, {format_pence(s[\'total\'])}"]\n    for r in s["rows"]:\n        line = f"  {r[\'category\']:<12}  {format_pence(r[\'spent\']):>11}"\n        if r["budget"] is not None:\n            line += f"  of {format_pence(r[\'budget\']):>9}  "\n            if r["left"] < 0:\n                line += f"OVER by {format_pence(-r[\'left\'])}"\n            else:\n                line += f"{format_pence(r[\'left\'])} left"\n        lines.append(line)\n    return lines\n\n\ndef build_parser():\n    parser = argparse.ArgumentParser(prog="spend", description="Track what you spend.")\n    parser.add_argument("--file", default="spend.json", help="the data file (default: spend.json)")\n    sub = parser.add_subparsers(dest="command", required=True)\n\n    p = sub.add_parser("add", help="record an expense")\n    p.add_argument("amount", help="for example 12.50 or £1,250")\n    p.add_argument("category", help="for example food")\n    p.add_argument("note", nargs="?", default="", help="what it was for")\n    p.add_argument("--date", default="", help="YYYY-MM-DD, DD/MM/YYYY, today or yesterday")\n\n    p = sub.add_parser("remove", help="remove an expense by its number")\n    p.add_argument("id", type=int)\n\n    p = sub.add_parser("list", help="list expenses, newest first")\n    p.add_argument("--month", help="YYYY-MM or MM/YYYY")\n    p.add_argument("--category")\n\n    p = sub.add_parser("summary", help="a month\'s spending by category")\n    p.add_argument("--month", help="YYYY-MM or MM/YYYY (default: this month)")\n\n    p = sub.add_parser("budget", help="show, set or remove monthly budgets")\n    p.add_argument("category", nargs="?")\n    p.add_argument("amount", nargs="?", help="an amount, or \'none\' to remove the budget")\n\n    p = sub.add_parser("import", help="import a bank statement CSV")\n    p.add_argument("statement")\n\n    p = sub.add_parser("export", help="export everything as CSV")\n    p.add_argument("out")\n    return parser\n\n\ndef cmd_add(data, args, today):\n    e = add_expense(data, args.amount, args.category, args.note, args.date, today)\n    print("Added", describe(e))\n    return True\n\n\ndef cmd_remove(data, args, today):\n    print("Removed", describe(remove_expense(data, args.id)))\n    return True\n\n\ndef cmd_list(data, args, today):\n    month = parse_month(args.month) if args.month else None\n    category = clean_category(args.category) if args.category else None\n    for line in expense_table(filter_expenses(data, month, category)):\n        print(line)\n    return False\n\n\ndef cmd_summary(data, args, today):\n    month = parse_month(args.month) if args.month else today.isoformat()[:7]\n    for line in summary_lines(month_summary(data, month)):\n        print(line)\n    return False\n\n\ndef cmd_budget(data, args, today):\n    if args.category is None:\n        if not data["budgets"]:\n            print("No budgets.")\n        for category, pence in sorted(data["budgets"].items()):\n            print(f"  {category:<12}  {format_pence(pence):>11} a month")\n        return False\n    if args.amount is None:\n        raise ValueError("give an amount, or \'none\' to remove the budget")\n    if args.amount.strip().lower() == "none":\n        category = clean_category(args.category)\n        remove_budget(data, category)\n        print(f"Removed the budget for {category}")\n    else:\n        category, pence = set_budget(data, args.category, args.amount)\n        print(f"Budget for {category}: {format_pence(pence)} a month")\n    return True\n\n\ndef cmd_import(data, args, today):\n    with open(args.statement, newline="", encoding="utf-8") as f:\n        result = import_statement(data, f.read())\n    print(f"Imported {result[\'added\']}, skipped {result[\'skipped\']} already there")\n    for problem in result["problems"]:\n        print(f"  could not read {problem}")\n    return result["added"] > 0\n\n\ndef cmd_export(data, args, today):\n    export_csv(data, args.out)\n    print(f"Exported {len(data[\'expenses\'])} expenses to {args.out}")\n    return False\n\n\nCOMMANDS = {"add": cmd_add, "remove": cmd_remove, "list": cmd_list, "summary": cmd_summary,\n            "budget": cmd_budget, "import": cmd_import, "export": cmd_export}\n\n\ndef main(argv=None, today=None):\n    """Run one command. Returns the exit status: 0 for success, 1 for an error."""\n    args = build_parser().parse_args(argv)\n    today = today or date.today()\n    try:\n        data = load_data(args.file)\n        if COMMANDS[args.command](data, args, today):\n            save_data(args.file, data)\n    except (ValueError, KeyError, OSError) as e:\n        message = e.args[0] if isinstance(e, KeyError) else str(e)\n        print(f"spend: {message}", file=sys.stderr)\n        return 1\n    return 0\n\n\nif __name__ == "__main__":\n    sys.exit(main())\n')

In [ ]:
import json
from datetime import date
import pytest
from spend import main

TODAY = date(2026, 11, 2)

@pytest.fixture
def run(tmp_path, capsys):
    path = str(tmp_path / "spend.json")
    def go(*argv):
        status = main(["--file", path, *argv], today=TODAY)
        out = capsys.readouterr()
        return status, out.out, out.err
    go.path = path
    return go

In [ ]:
# Check: run the exercise cell above first, then this one.
check("GOOD = open(\"spend.py\").read()\nBROKEN = {\n    \"never saves\": GOOD.replace(\"        if COMMANDS[args.command](data, args, today):\\n            save_data(args.file, data)\",\n                                \"        COMMANDS[args.command](data, args, today)\"),\n    \"errors on stdout\": GOOD.replace('print(f\"spend: {message}\", file=sys.stderr)', 'print(f\"spend: {message}\")'),\n    \"errors exit 0\": GOOD.replace(\"        return 1\\n    return 0\", \"        return 0\\n    return 0\"),\n    \"ignores --file\": GOOD.replace(\"load_data(args.file)\", 'load_data(\"spend.json\")').replace(\"save_data(args.file, data)\", 'save_data(\"spend.json\", data)'),\n    \"summary uses the wrong month\": GOOD.replace(\"else today.isoformat()[:7]\", 'else \"2026-10\"'),\n    \"list ignores the category\": GOOD.replace(\"filter_expenses(data, month, category)\", \"filter_expenses(data, month)\"),\n}\ndef test_catches():\n    \"your tests pass on the real program and catch all six broken versions\"\n    assert \"capsys\" in source\n    results = against(source, \"spend\", {\"correct\": GOOD, **BROKEN})\n    write_file(\"spend.py\", GOOD)\n    assert results[\"correct\"] == [], f\"these fail on the real program: {results['correct']}\"\n    missed = [k for k in BROKEN if not results[k]]\n    assert not missed, f\"not caught: {missed}\"")

### Quiz

Take this quiz on the website: it is marked for you and feeds your daily review.

## Where we are

All the modules are in the files below, now including `storage.py`. Today's code goes in `spend.py`, and it's the **only** module that prints or reads the command line. It has three jobs:

1. **Parse**: turn `["add", "12.50", "food", "lunch"]` into named values.
2. **Dispatch**: call the right core and storage functions, then save if the data changed.
3. **Present**: print the results for a person to read, and turn errors into messages.

It holds no rules about money or dates. If you find yourself checking an amount in `spend.py`, that check belongs in the core.

## Step 21: showing expenses

The `add` and `remove` commands confirm what they did in one line, and `list` prints a table. Columns line up using f-string widths from Week 4: `{x:>11}` right-aligns in 11 characters (right for numbers, so the pence line up), `{x:<12}` left-aligns (left for text).

```
   #  Date             Amount  Category      Note
   7  2026-11-02       £62.10  food          TESCO STORES 2041
   1  2026-11-01    £3,200.00  rent
      Total         £3,262.10  2 expenses
```

## Step 22: showing a summary

The summary prints yesterday's `month_summary` result. A person glancing at it wants three things: the month's total, the big categories, and any budget in trouble. So `OVER` is in capitals, at the end of its line where the eye lands.

```
November 2026: 6 expenses, £3,329.40
  rent            £3,200.00
  food               £65.90  of    £60.00  OVER by £5.90
  travel              £8.40  of    £50.00  £41.60 left
  fun                 £0.00  of    £30.00  £30.00 left
```

## Step 23: the parser

`argparse` (Week 21) turns the command line into named values, writes the `--help` text, and rejects nonsense with a usage message and exit status 2. **Subcommands**, like Git's `git add` and `git commit`, each get their own arguments:

| Command | Arguments |
|---|---|
| `add` | `amount`, `category`, an optional `note`, and `--date` (default `""`, meaning today) |
| `remove` | `id`, a whole number |
| `list` | optional `--month` and `--category` |
| `summary` | optional `--month` (default: this month) |
| `budget` | optional `category` and `amount` (no category lists the budgets; `none` as the amount removes one) |
| `import` | `statement`, a CSV file |
| `export` | `out`, a CSV file to write |

and before the command, `--file` (default `spend.json`) chooses the data file, so you can keep separate files or test without touching your real one.

## Step 24: commands and main

Each command gets a small function taking `(data, args, today)`. It does its job, prints the result, and returns `True` if it changed the data (so it needs saving) or `False` if it only looked. A dictionary from command names to functions replaces a long `if`/`elif` chain, as in Week 24's refactoring day.

`main` ties it together. Two details matter:

- It takes `argv` and `today` as parameters, defaulting to the real command line and the real date. Tests pass their own; the real program passes nothing. (`parse_args(None)` reads `sys.argv`.)
- It **returns** an exit status instead of calling `sys.exit` itself: 0 for success, 1 for an error. Only the last two lines of the file, which run when it's used as a script, pass that to `sys.exit`. Errors are printed to **standard error** as `spend: ...`, the convention for command-line tools (Week 21), so they don't get mixed into output someone is saving to a file.

## Trying it out

Here's a session with the finished `spend.py`, on 2 November 2026, starting with no data file. In your own `spend` folder you'd type exactly these lines.

```
$ python3 spend.py add 3,200 rent --date 01/11/2026
Added #1: £3,200.00 rent on 2026-11-01
$ python3 spend.py add 12.50 food 'lunch with Tola'
Added #2: £12.50 food (lunch with Tola) on 2026-11-02
$ python3 spend.py add 12.345 'Food & Drink' --date tomorrow
spend: not an amount: '12.345'; bad category: 'Food & Drink'; not a date: 'tomorrow'
$ python3 spend.py budget food 60
Budget for food: £60.00 a month
$ python3 spend.py budget fun 30
Budget for fun: £30.00 a month
$ python3 spend.py import statement.csv
Imported 7, skipped 0 already there
  could not read line 10: bad date '3O/10/2026'
  could not read line 11: bad amount '-1O.00'
$ python3 spend.py summary
November 2026: 6 expenses, £6,483.00
  rent            £6,400.00
  food               £74.60  of    £60.00  OVER by £14.60
  travel              £8.40
  fun                 £0.00  of    £30.00  £30.00 left
$ python3 spend.py list --category other
   #  Date             Amount  Category      Note
   8  2026-10-31        £3.80  other         CORNER CAFE
   9  2026-10-30        £6.45  other         PRET A MANGER
      Total            £10.25  2 expenses
$ python3 spend.py remove 99
spend: no expense #99
$ echo $?
1
```

Everything works as designed, and the session still shows a problem: **rent is counted twice**, £6,400 for November. It was typed in by hand on the 1st, and then the bank statement brought in the same payment as `CITY LETTINGS RENT`. The import's duplicate check compares notes, and the notes differ. No code is wrong; the design didn't foresee mixing typed and imported entries. That's normal, and it's exactly what using your own program for real turns up. Tomorrow's polish includes a way to deal with it.

## Step 25: testing from the outside

The core and storage have their own tests. The interface needs a few tests too, run the way a person uses it: give `main` a command line, then check what was printed, the exit status, and the file. These are **end-to-end** tests (Week 24, Day 1): slower and less precise than unit tests, so you need only a handful, but they're the only ones that check the layers are wired together correctly.

## Quiz

## Before you go

Put `spend.py` in your `spend` folder and use it for real for a day: add what you spend, set a budget or two, and run `summary`. Notice anything that's awkward to type or hard to read; tomorrow is for polish.